In [1]:
#2D Parallel Matrix Multiplication Using CUDA (Numba) in Python
from numba import cuda
import numpy as np
# CUDA Kernel
@cuda.jit
def matmul_2d(A, B, C):
    row, col = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        temp = 0.0
        for k in range(A.shape[1]):
            temp += A[row, k] * B[k, col]
        C[row, col] = temp


In [2]:
# Host Code
M, K, N = 4, 3, 5   # A: MxK, B: KxN, C: MxN
A = np.arange(M * K, dtype=np.float32).reshape(M, K)
B = np.arange(K * N, dtype=np.float32).reshape(K, N)
C = np.zeros((M, N), dtype=np.float32)


In [3]:
# Copy to GPU
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.to_device(C)
# 2D CUDA configuration
threads_per_block = (16, 16)
blocks_per_grid_x = (N + threads_per_block[1] - 1)
threads_per_block[1]
blocks_per_grid_y = (M + threads_per_block[0] - 1)
threads_per_block[0]
blocks_per_grid = (blocks_per_grid_y, blocks_per_grid_x)

In [6]:
# Copy to GPU
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.to_device(C)
# 2D CUDA configuration
threads_per_block = (16, 16)
blocks_per_grid_x = (N + threads_per_block[1] - 1)
threads_per_block[1]
blocks_per_grid_y = (M + threads_per_block[0] - 1)
threads_per_block[0]
blocks_per_grid = (blocks_per_grid_y, blocks_per_grid_x)
# Launch kernel
matmul_2d[blocks_per_grid, threads_per_block](d_A, d_B, d_C)
cuda.synchronize()
# Copy result back
C = d_C.copy_to_host()
print("Matrix A:\n", A)
print("Matrix B:\n", B)
print("Matrix C = A x B:\n", C)

Matrix A:
 [[ 0.  1.  2.]
 [ 3.  4.  5.]
 [ 6.  7.  8.]
 [ 9. 10. 11.]]
Matrix B:
 [[ 0.  1.  2.  3.  4.]
 [ 5.  6.  7.  8.  9.]
 [10. 11. 12. 13. 14.]]
Matrix C = A x B:
 [[ 25.  28.  31.  34.  37.]
 [ 70.  82.  94. 106. 118.]
 [115. 136. 157. 178. 199.]
 [160. 190. 220. 250. 280.]]


# CNN

In [8]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device name: Tesla T4


In [9]:
print("Starting test...")

import torch
import torch.nn as nn
import torch.nn.functional as F

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Define CNN
class MyCNN(nn.Module):
    def __init__(self):
        super(MyCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.LazyLinear(1)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = torch.flatten(x, 1)
        x = torch.sigmoid(self.fc1(x))
        return x

# Create model and move to GPU
model = MyCNN().to(device)
print("Model created and moved to device")

# Create input tensor and move to GPU
x = torch.randn(1, 3, 128, 128).to(device)

# Forward pass
with torch.no_grad():   # disables gradient calculation (faster inference)
    out = model(x)

print("Output shape:", out.shape)

Starting test...
Using device: cuda
GPU: Tesla T4
Model created and moved to device
Output shape: torch.Size([1, 1])
